In [0]:
pip install tensorflow


In [0]:
import mlflow
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd
import numpy as np

dbutils.widgets.text("catalog_name", "fma_catalog", "Unity Catalog Name")
catalog = dbutils.widgets.get("catalog_name")

df = spark.table(f"{catalog}.silver.audio_features").toPandas()

mfcc_df = pd.DataFrame(df['mfcc_means'].tolist(), columns=[f'mfcc_{i}' for i in range(13)])
X = pd.concat([df[['tempo', 'spectral_centroid_mean']], mfcc_df], axis=1)

# Encode Labels (Genres)
le = LabelEncoder()
y = le.fit_transform(df['genre_top'])
num_classes = len(le.classes_)

# Train/Test Split & Scaling
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# ANN Architecture
def build_model():
    model = tf.keras.Sequential([
        layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer='adam', 
                  loss='sparse_categorical_crossentropy', 
                  metrics=['accuracy'])
    return model

mlflow.set_experiment(f"/Users/{dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()}/fma_genre_classification")
mlflow.tensorflow.autolog()

with mlflow.start_run(run_name="Initial_ANN_Run"):
    model = build_model()
    history = model.fit(X_train, y_train, 
                        epochs=50, 
                        batch_size=8, 
                        validation_data=(X_test, y_test))
    
    # Log the label encoder for inference later
    mlflow.log_dict({"classes": le.classes_.tolist()}, "classes.json")
    
    print(f"Training Complete. Best Val Accuracy: {max(history.history['val_accuracy']):.2f}")